# Day 40 — LangGraph II: cycles & memory

Real agents **loop** and **remember**. Conditional edges create cycles; a **checkpointer**
persists state across calls (per `thread_id`).

```bash
pip install langgraph
```
> ✅ Runs offline — a counter that ticks until it reaches 3.

In [ ]:
# ▶ Run this first — puts the repo root on sys.path so `shared/` imports work.
import sys, pathlib
root = pathlib.Path.cwd()
while root != root.parent and not (root / "shared" / "llm.py").exists():
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
print("repo root on sys.path:", root)

## 🔬 Your turn

Fill in the `TODO`s, then run. Solution below.

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

class State(TypedDict):
    count: int

def tick(state: State) -> dict:
    return {"count": state["count"] + 1}

def route(state: State):
    return END if state["count"] >= 3 else "tick"

builder = StateGraph(State)
builder.add_node("tick", tick)
builder.add_edge(START, "tick")
builder.add_conditional_edges("tick", route, {"tick": "tick", END: END})

# TODO: graph = builder.compile(checkpointer=MemorySaver())
#       print(graph.invoke({"count": 0}, config={"configurable": {"thread_id": "a"}}))


## 🔒 Solution

Idiomatic, current-API version.

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

class State(TypedDict):
    count: int

def tick(state: State) -> dict:
    return {"count": state["count"] + 1}

def route(state: State):
    return END if state["count"] >= 3 else "tick"

builder = StateGraph(State)
builder.add_node("tick", tick)
builder.add_edge(START, "tick")
builder.add_conditional_edges("tick", route, {"tick": "tick", END: END})

graph = builder.compile(checkpointer=MemorySaver())
print(graph.invoke({"count": 0}, config={"configurable": {"thread_id": "a"}}))